# Week 3 Practice Tutorial: Feedforward Networks on MNIST

**MATH3881 / MATH5881, UNSW.** Instructor: Sarat Moka.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saratmoka/MATH3881-5881/blob/main/tutorials/week03/Week3_Practice_Tutorial.ipynb)

Today you will do two things.

**Problem 1.** Train your first feedforward neural network on MNIST, step by step, in PyTorch. You will see loss curves, overfitting, and what each piece of the training loop is doing.

**Problem 2.** Test the *dropout-as-ensemble* claim empirically. Train one network with dropout, then separately train a handful of small thinned subnetworks and average their predictions. You should find that the two strategies give comparable test accuracy, which is the heart of why dropout works.

The notebook is designed for Google Colab (no local install required). A CPU runtime is enough; GPU just makes it faster. Estimated total running time on CPU: about 5 minutes.


## Setup


We import PyTorch, torchvision (for MNIST), matplotlib, and numpy. We detect whether a GPU is available, and seed every random number generator the notebook touches so that re-running gives exactly the same numbers.

There are five separate sources of randomness in a typical training run: Python's built-in `random`, NumPy's RNG, PyTorch's CPU RNG, PyTorch's CUDA RNG, and cuDNN's choice of which kernel to use. We pin all five below.


In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Seed every random number generator we touch, so the notebook is reproducible.
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Make cuDNN pick deterministic algorithms (slight speed cost; matters only on GPU).
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


---

## Problem 1.  A feedforward network on MNIST, step by step


### 1.1 Load the MNIST dataset

MNIST is a collection of 70,000 hand-written digit images (60,000 train, 10,000 test), each $28 \times 28$ grayscale, with labels in $\{0, 1, \ldots, 9\}$. Our network takes a flat vector of length $784$, so we need to flatten each image and normalise pixel values to $[0, 1]$. PyTorch handles both through a *transform pipeline* that runs automatically every time a sample is loaded.


**Step 1: the transform pipeline.** A torchvision transform is a callable that maps one sample to another. We chain several together with `transforms.Compose([...])`, which applies each transform left-to-right (function composition: the output of one is the input of the next). For MNIST we need two steps.

* `transforms.ToTensor()` converts a PIL image (8-bit integers $0, 1, \ldots, 255$) to a float tensor in $[0, 1]$ with shape `(1, 28, 28)`. The division by $255$ happens automatically; this is the normalisation step.
* `transforms.Lambda(lambda x: x.view(-1))` reshapes that `(1, 28, 28)` tensor into a length-784 vector. This is the flatten step needed because our model is a feedforward (not convolutional) network.

The composed `transform` is attached to the dataset object; each sample passes through it on the fly when loaded.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                    # PIL image -> float tensor in [0, 1], shape (1, 28, 28)
    transforms.Lambda(lambda x: x.view(-1)),  # flatten to a length-784 vector
])


**Step 2: load the train and test datasets.** `torchvision.datasets.MNIST` provides the standard dataset. The arguments are:

* `'./data'`: where to store the dataset files on disk. On the first run the files are downloaded; on subsequent runs they are read from cache.
* `train=True / train=False`: MNIST ships with two fixed partitions, a 60,000-sample training set and a 10,000-sample test set. We load both. The split between them was set by the dataset providers; we do not get to choose it. The convention of separating train from test is universal in supervised learning: anything used during training cannot also be used to evaluate generalisation.
* `download=True`: download if not present locally (a few megabytes; takes seconds on Colab).
* `transform=transform`: apply the pipeline above to every sample.


In [ ]:
train_full = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_set   = datasets.MNIST('./data', train=False, download=True, transform=transform)

print(f'Official train set: {len(train_full):,}    Official test set: {len(test_set):,}')


**Step 3: carve a small training set and a validation set out of the official training set.** During training we want to monitor whether the model is generalising, but we are *not* allowed to look at the test set. Any decision we make based on the test set (when to stop, which hyperparameter to pick, which architecture to use) leaks information from it into the model, and the final test accuracy is no longer an honest estimate of generalisation. The standard fix is to carve a *validation* set out of the official training data and keep the test set sealed until the very end.

We use `torch.utils.data.random_split` to partition `train_full` into a 5,000-sample training set, a 10,000-sample validation set, and a 45,000-sample remainder that we discard. The split is deterministic given our seed.

Why only 5,000 training samples when 60,000 are available? **Pedagogical choice.** With the full training set, MNIST is so easy that even a small feedforward network reaches $97\%+$ test accuracy without any regularisation, and we never get to see the regularisers earn their keep. Training on only 5,000 samples makes the network overfit visibly: train accuracy will climb near $100\%$ while test accuracy lags well behind, and that gap is what dropout (and friends) are there to close. In a real project you would use all the data you can get.


In [ ]:
train_set, val_set, _discard = random_split(
    train_full,
    [5_000, 10_000, 45_000],
    generator=torch.Generator().manual_seed(SEED),
)


**Step 4: batching with `DataLoader`.** A `DataLoader` wraps a dataset and yields mini-batches. We use:

* **Training**: batch size 128, `shuffle=True`. Shuffling each epoch decorrelates the gradient updates across passes and helps generalisation.
* **Validation and test**: batch size 512, `shuffle=False`. Larger batches are faster (no backprop needed for evaluation) and shuffling has no effect on the result.


In [ ]:
# Explicit generator on the train loader makes the per-epoch shuffle reproducible.
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
val_loader   = DataLoader(val_set,   batch_size=512, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=512, shuffle=False)

print(f'Train: {len(train_set):,}    Val: {len(val_set):,}    Test: {len(test_set):,}')


A look at a few training digits with their labels.


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, (x, y) in zip(axes.flat, train_set):
    ax.imshow(x.view(28, 28), cmap='gray')
    ax.set_title(f'{y}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()


### 1.2 Define the model

Our architecture: a feedforward network with two hidden layers,
$$ 784 \;\xrightarrow{\;W^{[1]}, b^{[1]}\;}\; 128 \;\xrightarrow{\mathrm{ReLU}}\; 128 \;\xrightarrow{\;W^{[2]}, b^{[2]}\;}\; 64 \;\xrightarrow{\mathrm{ReLU}}\; 64 \;\xrightarrow{\;W^{[3]}, b^{[3]}\;}\; 10. $$
This is exactly the kind of network from Week 2, written in PyTorch building blocks. Each block is a Python object:

* `nn.Linear(in_features, out_features)` is the affine map $x \mapsto Wx + b$. PyTorch creates the weight matrix $W$ (shape `(out_features, in_features)`) and bias vector $b$ as *trainable parameters* automatically; you do not initialise them by hand. The PyTorch default initialisation is close to the Xavier scheme from Week 2 §6.
* `nn.ReLU()` applies $\max(0, \cdot)$ elementwise. It has no parameters; it is just a function masquerading as a module.
* `nn.Sequential(layer_1, layer_2, \ldots)` is a container that chains its arguments. Calling `model(x)` runs $x$ through them left-to-right; calling `model.parameters()` returns all trainable parameters across the chain. This is what lets the optimiser update *every* weight matrix in one call.

One subtlety: the final layer outputs raw scores (called *logits*), not probabilities. We do not append a softmax. The reason is numerical: `nn.CrossEntropyLoss` (Section 1.3) applies log-softmax internally with a numerically stable formula, and applying softmax twice would be wasteful and unstable.

After defining the model we call `.to(device)`, which moves all parameter tensors to the GPU if one is available, or leaves them on CPU otherwise. Subsequent inputs `x` also need to be moved to the same device.


In [ ]:
def make_model(hidden1=128, hidden2=64, dropout_p=0.0):
    layers = [
        nn.Linear(784, hidden1),
        nn.ReLU(),
    ]
    if dropout_p > 0:
        layers.append(nn.Dropout(dropout_p))
    layers += [
        nn.Linear(hidden1, hidden2),
        nn.ReLU(),
    ]
    if dropout_p > 0:
        layers.append(nn.Dropout(dropout_p))
    layers += [nn.Linear(hidden2, 10)]
    return nn.Sequential(*layers)

model = make_model().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Trainable parameters: {n_params:,}')


### 1.3 Loss function and optimiser

Two PyTorch objects do the heavy lifting during training.

**`nn.CrossEntropyLoss`** is the softmax cross-entropy loss from Week 1. For a single sample with true class $y \in \{0, \ldots, 9\}$ and predicted logits $z \in \mathbb{R}^{10}$, it computes
$$ \ell(z, y) \;=\; -\log \frac{e^{z_y}}{\sum_{k=0}^{9} e^{z_k}} \;=\; -z_y + \log \sum_k e^{z_k}, $$
the negative log-probability that the network assigns to the correct class. It accepts logits (not probabilities) precisely so that the log-sum-exp on the right can be evaluated with the numerically stable trick (subtracting $\max_k z_k$ inside the exponentials).

**`optim.SGD`** implements gradient descent. We pass it `model.parameters()` (every weight matrix and bias vector that needs updating) together with two knobs:

* `lr=0.01` is the learning rate $\alpha$ from Week 3 §1, the step size in $\theta \leftarrow \theta - \alpha\, \nabla C(\theta)$.
* `momentum=0.9` is a mild improvement on plain GD that we will study properly in Week 4. For now, think of it as smoothing the gradient direction across iterations.

After this cell, the optimiser holds *references* to the model's parameters. Calling `optimizer.step()` updates them in place using whatever gradients are currently sitting on them.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)


### 1.4 Training and evaluation loops

Training a network in PyTorch is the same five-line dance, repeated for every mini-batch:

1. `optimizer.zero_grad()` — set the gradient of every parameter to zero. By default PyTorch *accumulates* gradients into the `.grad` attribute of each parameter, so if you forget this step, you are effectively summing gradients across batches.
2. `logits = model(x)` — forward pass. This runs the input through every layer in `nn.Sequential`.
3. `loss = criterion(logits, y)` — evaluate the loss as a scalar tensor.
4. `loss.backward()` — backpropagation, computed automatically by PyTorch's autograd. We will not implement this by hand in this course, but conceptually it walks the computational graph backwards and fills in $\partial \text{loss} / \partial \theta$ for every learnable parameter $\theta$ (see Week 3, §3-4 for the underlying algorithm).
5. `optimizer.step()` — take one gradient descent step using those gradients.

Two more details show up in the helpers below.

* `model.train()` and `model.eval()` switch the model between training mode and evaluation mode. The behaviour of some layers (notably `nn.Dropout` and `nn.BatchNorm`) differs between the two: dropout zeros activations only in train mode; batch norm uses batch statistics in train mode and a running average in eval mode.
* The `@torch.no_grad()` decorator on `evaluate` tells PyTorch not to build a computational graph for the forward pass, which saves memory and time. Use it whenever you only want predictions, not gradients.

Both functions return the per-sample loss (so values from different batch sizes are comparable) and the classification accuracy.


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total_count = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_count   += x.size(0)
    return total_loss / total_count, total_correct / total_count

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_count = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss    += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_count   += x.size(0)
    return total_loss / total_count, total_correct / total_count


### 1.5 Train the model

We wrap everything in a `fit` function that loops over epochs. An *epoch* is one full pass through the training set: with 5,000 samples and batch size 128, the train loader yields $\lceil 5{,}000 / 128 \rceil = 40$ batches per epoch, and we run `train_one_epoch` on all of them. After each epoch we also evaluate on the validation set, so we can watch how the network is doing on data it has not seen during the gradient updates.

We train for 60 epochs. With only 5,000 training samples, an unregularised feedforward network has enough capacity to essentially memorise the training set, and the training accuracy approaches $100\%$. The validation loss, however, plateaus partway through and then starts to creep upward; that movement, validation loss going up while training loss keeps going down, is the textbook signature of overfitting.


In [ ]:
def fit(model, n_epochs=60, lr=0.01, momentum=0.9,
        train_loader=train_loader, val_loader=val_loader):
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)
        print(f'epoch {epoch:2d}  train loss {tr_loss:.4f}  val loss {va_loss:.4f}  '
              f'train acc {tr_acc:.4f}  val acc {va_acc:.4f}')
    return history

torch.manual_seed(SEED)
model_baseline = make_model().to(device)
hist_baseline = fit(model_baseline, n_epochs=60)


### 1.6 Plot the training curves


In [ ]:
def plot_history(histories, names, title=''):
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
    for h, name in zip(histories, names):
        epochs = range(1, len(h['train_loss']) + 1)
        ax_loss.plot(epochs, h['train_loss'], label=f'{name} train', linestyle='--')
        ax_loss.plot(epochs, h['val_loss'],   label=f'{name} val')
        ax_acc.plot(epochs, h['train_acc'],   label=f'{name} train', linestyle='--')
        ax_acc.plot(epochs, h['val_acc'],     label=f'{name} val')
    ax_loss.set_xlabel('epoch'); ax_loss.set_ylabel('loss'); ax_loss.legend(fontsize=8)
    ax_acc.set_xlabel('epoch'); ax_acc.set_ylabel('accuracy'); ax_acc.legend(fontsize=8)
    if title:
        fig.suptitle(title); fig.tight_layout()
    plt.show()

plot_history([hist_baseline], ['baseline'], title='Baseline FFN on MNIST')


### 1.7 Final test accuracy

Now, and only now, we touch the test set. It was sealed during training (we never used it to choose anything: not the architecture, not the learning rate, not the number of epochs). Its accuracy is our honest estimate of how well the model would do on completely new digits.

We will also print the train accuracy at the last epoch and the *generalisation gap* (train minus test). A large positive gap means the model has fit the training data better than the test data, that is, it has memorised some patterns specific to the training set that do not transfer.


In [ ]:
test_loss_baseline, test_acc_baseline = evaluate(model_baseline, test_loader, criterion)
print(f'Baseline test accuracy: {test_acc_baseline:.4f}  (test loss {test_loss_baseline:.4f})')
print(f'Train accuracy at last epoch: {hist_baseline["train_acc"][-1]:.4f}')
print(f'Generalisation gap (train acc minus test acc): '
      f'{hist_baseline["train_acc"][-1] - test_acc_baseline:+.4f}')


The gap between train and test accuracy is the network *over-fitting* the training data. Problem 2 attacks it from the dropout side. First, three small experiments.


> ### <font color="#1b5e20">1.8 Tasks</font>
>
> Two small experiments. Run them, look at the numbers, then think about what you observed.
>
> <font color="#1b5e20">**Task 1.8.1: change the seed.**</font> Pick another value for `SEED` (try 1, 42, 123) and re-run sections 1.5 through 1.7. How much does the final test accuracy move from run to run? If the seed-to-seed spread is comparable to the dropout-vs-baseline gap, you cannot conclude that dropout helps from a single comparison; you would need to average over seeds.
>
> <font color="#1b5e20">**Task 1.8.2: train on Fashion-MNIST.**</font> Fashion-MNIST is a drop-in replacement for MNIST with the same $28 \times 28$ grayscale shape and the same 10 classes, but the labels are clothing items (t-shirt, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot) instead of digits. It is significantly harder than MNIST: the same architecture and recipe typically lands around 80% test accuracy instead of 93%+. In the code cell below, load Fashion-MNIST using `datasets.FashionMNIST` (same constructor signature as `datasets.MNIST`), build a 5,000 / 10,000 train/val split and corresponding DataLoaders, then re-train the baseline model and report the test accuracy alongside the MNIST baseline number. The point is to see that the same pipeline behaves very differently when the data is harder.


In [ ]:
# Task 1.8.2: train the baseline model on Fashion-MNIST.

fmnist_train_full = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
fmnist_test_set   = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)

fmnist_train_set, fmnist_val_set, _fmnist_discard = random_split(
    fmnist_train_full,
    [5_000, 10_000, 45_000],
    generator=torch.Generator().manual_seed(SEED),
)

fmnist_train_loader = DataLoader(
    fmnist_train_set, batch_size=128, shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)
fmnist_val_loader  = DataLoader(fmnist_val_set,  batch_size=512, shuffle=False)
fmnist_test_loader = DataLoader(fmnist_test_set, batch_size=512, shuffle=False)

torch.manual_seed(SEED)
model_fmnist = make_model().to(device)
hist_fmnist = fit(model_fmnist, n_epochs=60,
                  train_loader=fmnist_train_loader,
                  val_loader=fmnist_val_loader)
test_loss_fmnist, test_acc_fmnist = evaluate(model_fmnist, fmnist_test_loader, criterion)
print(f'MNIST baseline test accuracy:        {test_acc_baseline:.4f}')
print(f'Fashion-MNIST baseline test accuracy: {test_acc_fmnist:.4f}  (test loss {test_loss_fmnist:.4f})')



---

## Problem 2.  Dropout as an ensemble of thinned networks

Recall (Week 2, §6) that dropout with keep-probability $p$ randomly zeros each hidden unit's activation, independently, each forward pass. The standard story is:

> *Training one network of width $W$ with dropout is approximately equivalent to training an ensemble of $2^W$ thinned subnetworks (each picking a random subset of units) that share parameters; at test time, dropout's prediction is approximately the average of all thinned subnetworks' predictions.*

We cannot afford to enumerate $2^W$ subnetworks. But we can sample $K$ of them, train each one separately on the same data, average their softmax outputs at test time, and compare to a single dropout-trained network. The claim is that the two should land in the same ballpark.

We will compare three things:

1. **Baseline** (no regularisation) from Problem 1.
2. **One dropout net**: one network of width 128 trained with $\mathrm{Dropout}(0.5)$.
3. **Ensemble of $K$ thinned nets**: $K$ independent networks of half the width (64 hidden units each), no dropout, trained on the same data. At test time, we average their softmax probabilities.


### 2.1 Train one network with dropout

**What `nn.Dropout(p)` actually does.** During *training*, every time a tensor passes through this layer, each entry is independently set to zero with probability $p$ (the *drop* probability) and kept with probability $1-p$. The surviving entries are multiplied by $1/(1-p)$ so that the expected sum of activations is unchanged: this is the *inverted dropout* trick from Week 2 §6. During *evaluation* (`model.eval()`), dropout is the identity, no zeros, no scaling. This is why `model.train()` and `model.eval()` matter.

We add a dropout layer after each ReLU, with keep-probability $1-p = 0.5$.


In [ ]:
torch.manual_seed(SEED)
model_dropout = make_model(hidden1=128, hidden2=64, dropout_p=0.5).to(device)
hist_dropout  = fit(model_dropout, n_epochs=60)
test_loss_dropout, test_acc_dropout = evaluate(model_dropout, test_loader, criterion)
print(f'Dropout test accuracy: {test_acc_dropout:.4f}')


### 2.2 Train $K$ thinned subnetworks

Each thinned network has half the hidden width of the dropout net (a rough match: dropout with $p=0.5$ keeps half the units in expectation). We train $K = 5$ of them, each with a different random initialisation. There is no dropout in the thinned nets.


In [ ]:
K = 5
thinned_models = []
for k in range(K):
    torch.manual_seed(SEED + 1 + k)  # different init each
    m = make_model(hidden1=64, hidden2=32, dropout_p=0.0).to(device)
    print(f'--- Thinned net {k + 1} of {K} ---')
    fit(m, n_epochs=60)
    thinned_models.append(m)

print(f'Trained {len(thinned_models)} thinned subnetworks.')


### 2.3 Ensemble prediction

At test time, each thinned net returns logits; we apply softmax, average the resulting probability vectors across the $K$ networks, then take the argmax. This is the standard ensemble rule.


In [ ]:
@torch.no_grad()
def ensemble_predict(models, loader):
    """Return overall test accuracy of the softmax-averaged ensemble."""
    for m in models:
        m.eval()
    total_correct, total_count = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        probs = torch.zeros(x.size(0), 10, device=device)
        for m in models:
            probs += torch.softmax(m(x), dim=1)
        probs /= len(models)
        preds = probs.argmax(dim=1)
        total_correct += (preds == y).sum().item()
        total_count   += x.size(0)
    return total_correct / total_count

test_acc_ensemble = ensemble_predict(thinned_models, test_loader)
print(f'Ensemble test accuracy (K = {K} thinned nets): {test_acc_ensemble:.4f}')


### 2.4 Side-by-side comparison

Three test accuracies: the unregularised baseline, the single dropout network, and the ensemble of $K$ thinned subnetworks.


In [ ]:
thinned_individual_accs = []
for k, m in enumerate(thinned_models):
    _, acc = evaluate(m, test_loader, criterion)
    thinned_individual_accs.append(acc)

print('Test accuracies:')
print(f'  baseline (no regularisation):   {test_acc_baseline:.4f}')
print(f'  one net with dropout(0.5):      {test_acc_dropout:.4f}')
print(f'  best individual thinned net:    {max(thinned_individual_accs):.4f}')
print(f'  mean individual thinned net:    {np.mean(thinned_individual_accs):.4f}')
print(f'  ensemble of {K} thinned nets:    {test_acc_ensemble:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
labels = ['baseline', 'dropout', 'thinned\n(best)', 'thinned\n(mean)', f'ensemble\n(K={K})']
values = [test_acc_baseline, test_acc_dropout, max(thinned_individual_accs),
          float(np.mean(thinned_individual_accs)), test_acc_ensemble]
bars = ax.bar(labels, values, color=['gray', 'C0', 'C2', 'C2', 'C3'])
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.001, f'{v:.4f}',
            ha='center', va='bottom', fontsize=9)
ax.set_ylim(min(values) - 0.01, max(values) + 0.01)
ax.set_ylabel('test accuracy')
ax.set_title('Dropout vs ensemble of thinned networks')
plt.tight_layout(); plt.show()


### 2.5 What did we just see?

Read off the bar chart. Three things to notice.

* **The baseline overfits.** Its training accuracy reached close to $100\%$ but its test accuracy is significantly lower; this is the gap that regularisation is meant to close.
* **The single dropout net closes that gap.** Trained on the same 5,000 samples with the same architecture, but with `nn.Dropout(0.5)` between the hidden layers, it generalises better than the baseline. The randomness during training prevents the net from memorising any single training pattern too aggressively.
* **The ensemble of $K$ thinned subnetworks closes the gap by another route, and ends up at a similar test accuracy to the dropout net.** Each thinned net individually has lower capacity and underfits a bit; averaging $K$ of them cancels the idiosyncratic errors of each one.

That last point is the empirical content of the dropout-as-ensemble claim. Dropout buys you the variance-reduction benefit of ensembling *without* having to train, store, and run $K$ separate networks. One parameter vector implicitly carries the averaged behaviour of an exponentially large family of thinned subnetworks; at test time, a single forward pass through the full network is approximately the ensemble's averaged prediction.


> ### <font color="#1b5e20">2.6 Tasks</font>
>
> Two tasks. The second is the most interesting thing in the notebook; do not skip it.
>
> <font color="#1b5e20">**Task 2.6.1: change the dropout rate.**</font> Re-run section 2.1 with $p = 0.2$ and then $p = 0.8$. Plot the validation curves alongside the original $p = 0.5$ result. Does a higher dropout rate always help? Why does $p = 0.8$ hurt? Connect to the Week 2 §6 view that dropout effectively reduces the capacity used per forward pass by a factor of $1 - p$.
>
> <font color="#1b5e20">**Task 2.6.2: Monte Carlo dropout at test time.**</font> When we evaluated the dropout net in section 2.1 we called `model.eval()`, which switches `nn.Dropout` into pass-through mode and gives us one deterministic forward pass. Try the *opposite*: put the trained `model_dropout` into train mode (`model_dropout.train()`), then push the test set through it $T = 20$ times, each time with a fresh random dropout mask, and average the $T$ softmax outputs before taking the argmax. (Hint: this is essentially `ensemble_predict` from section 2.3 but with *one* trained network instead of $K$ separate ones; use the `@torch.no_grad()` decorator to skip gradient tracking, but do *not* call `model.eval()`.) You should now have three test accuracies for comparison: the standard dropout-off test pass from section 2.1, the $K$-thinned-net ensemble from section 2.4, and this new Monte Carlo dropout number. All three should land within a fraction of a percent of each other. This is the dropout-as-ensemble equivalence made empirical from a fresh angle: the trained dropout network already carries the ensemble *inside* it, and you can recover the averaged prediction by sampling subnetworks at test time, no separate networks required. (For the curious: this trick is called *Monte Carlo dropout*; Gal and Ghahramani (2016) use it to get cheap predictive uncertainty estimates from deep networks.)


In [ ]:
# Task 2.6.2: Monte Carlo dropout at test time.

@torch.no_grad()
def mc_dropout_predict(model, loader, T=20):
    model.train()
    total_correct, total_count = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        probs = torch.zeros(x.size(0), 10, device=device)
        for _ in range(T):
            probs += torch.softmax(model(x), dim=1)
        probs /= T
        preds = probs.argmax(dim=1)
        total_correct += (preds == y).sum().item()
        total_count += x.size(0)
    return total_correct / total_count

acc_mc = mc_dropout_predict(model_dropout, test_loader, T=20)
print(f'Standard dropout test accuracy:      {test_acc_dropout:.4f}')
print(f'Ensemble of thinned nets accuracy:   {test_acc_ensemble:.4f}')
print(f'Monte Carlo dropout accuracy (T=20): {acc_mc:.4f}')

